# Phase 2, Workshop 03 — Player Tokens on the Board

Your board looks great, but there are no players on it yet! In this workshop you will add coloured circular tokens that represent each player's position. You will learn how to place tokens inside cells, handle two players on the same cell, and prepare the `updateTokens` function that refreshes token positions every time the state changes.

**What you will learn:**
- Creating and styling circular token elements
- Placing tokens inside the correct board cell using the cell's ID
- Clearing and re-rendering tokens when positions change
- CSS transitions for smooth token movement

**Time:** roughly 15 to 20 minutes.

## Section 1 — Token Design

Each player token is a small coloured circle with their player number inside. Player 1 is red, Player 2 is blue. The CSS is straightforward:

```css
.token {
    width: 14px;
    height: 14px;
    border-radius: 50%;           /* makes it circular */
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 0.5rem;
    font-weight: 900;
    color: #fff;
    box-shadow: 0 1px 3px rgba(0,0,0,0.4);
}

.token-p1 { background: var(--p1); }   /* red */
.token-p2 { background: var(--p2); }   /* blue */
```

## Section 2 — Token Containers in Each Cell

Each cell has a small `<div>` with the class `tokens` and an ID like `tokens-0`, `tokens-5`, etc. This is where we will insert the player token elements. We already built the cells in Workshop 02, so we just need to add one more line to each cell's HTML:

```javascript
html += '<div class="tokens" id="tokens-' + i + '"></div>';
```

The `tokens` container sits at the bottom of the cell and uses flexbox to arrange multiple tokens side by side (for when both players land on the same space).

## Section 3 — The updateTokens Function

The key function clears all token containers, then places tokens in the correct cells based on the current game state.

In [ ]:
import os

project_folder = os.path.join(os.path.expanduser("~"), "text_monopoly", "phase2")
os.makedirs(project_folder, exist_ok=True)

html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Phase 2 - Player Tokens</title>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Outfit:wght@400;600;700;900&display=swap');

        :root {
            --bg: #1a1520; --board-bg: #f5f0e8; --board-border: #1a1520;
            --board-center: #dcd4c8; --text-dark: #1a1520;
            --p1: #e84855; --p2: #3185fc; --gold: #f2a93b;
            --brown: #8B4513; --sky: #87CEEB; --pink: #d63384;
            --orange: #fd7e14; --red: #e84855; --yellow: #ffc107;
            --dgreen: #198754; --navy: #0d6efd; --station: #444; --utility: #aaa;
        }
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { font-family: 'Outfit', sans-serif; background: var(--bg); display: flex; flex-direction: column; align-items: center; padding: 20px; color: var(--text-dark); }
        h1 { color: #f5f0e8; margin-bottom: 12px; font-weight: 900; }

        .board { display: grid; grid-template-columns: 72px repeat(9, 56px) 72px; grid-template-rows: 72px repeat(9, 56px) 72px; gap: 1px; background: var(--board-border); border: 3px solid var(--board-border); border-radius: 8px; overflow: hidden; font-size: 0.5rem; }
        .cell { background: var(--board-bg); color: var(--text-dark); display: flex; flex-direction: column; align-items: center; justify-content: center; position: relative; padding: 2px; text-align: center; line-height: 1.15; font-weight: 600; overflow: hidden; }
        .cell .color-bar { position: absolute; width: 100%; height: 8px; left: 0; }
        .side-top .color-bar { bottom: 0; }
        .side-bottom .color-bar { top: 0; }
        .side-left .color-bar { top: 0; left: unset; right: 0; width: 8px; height: 100%; }
        .side-right .color-bar { top: 0; left: 0; width: 8px; height: 100%; }
        .cell .name { font-size: 0.45rem; font-weight: 700; z-index: 1; }
        .cell .price { font-size: 0.4rem; color: #666; z-index: 1; }
        .cell .icon { font-size: 0.9rem; z-index: 1; }
        .corner { font-size: 0.55rem; font-weight: 900; }
        .board-center { grid-column: 2 / 11; grid-row: 2 / 11; background: var(--board-center); display: flex; flex-direction: column; align-items: center; justify-content: center; gap: 8px; }
        .board-center .title { font-size: 1.8rem; font-weight: 900; letter-spacing: -1px; }

        /* Token styles */
        .tokens { display: flex; gap: 2px; z-index: 2; margin-top: 2px; }
        .token { width: 14px; height: 14px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 0.5rem; font-weight: 900; color: #fff; box-shadow: 0 1px 3px rgba(0,0,0,0.4); transition: all 0.4s cubic-bezier(0.34, 1.56, 0.64, 1); }
        .token-p1 { background: var(--p1); }
        .token-p2 { background: var(--p2); }

        /* Controls below the board */
        .controls { margin-top: 16px; display: flex; gap: 10px; }
        .controls button { font-family: 'Outfit', sans-serif; padding: 10px 24px; border: none; border-radius: 8px; font-weight: 700; font-size: 0.9rem; cursor: pointer; color: #fff; }
        .btn-p1 { background: var(--p1); }
        .btn-p2 { background: var(--p2); }
        .btn-p1:hover { filter: brightness(1.1); }
        .btn-p2:hover { filter: brightness(1.1); }
        .info { color: #f5f0e8; margin-top: 8px; font-size: 0.85rem; }
    </style>
</head>
<body>
    <h1>Player Tokens Demo</h1>
    <div class="board" id="board"></div>
    <div class="controls">
        <button class="btn-p1" onclick="movePlayer(0)">Move P1 (roll)</button>
        <button class="btn-p2" onclick="movePlayer(1)">Move P2 (roll)</button>
    </div>
    <div class="info" id="info">Both players start on GO. Click to move.</div>

    <script>
        const B = [
            {n:"GO",t:"go",icon:"🏁"},{n:"Old Kent Rd",t:"prop",c:60,col:"var(--brown)"},
            {n:"Chest",t:"chest",icon:"📦"},{n:"Whitechapel",t:"prop",c:60,col:"var(--brown)"},
            {n:"Income Tax",t:"tax",icon:"💸"},{n:"Kings Cross",t:"prop",c:200,col:"var(--station)"},
            {n:"Angel",t:"prop",c:100,col:"var(--sky)"},{n:"Chance",t:"chance",icon:"❓"},
            {n:"Euston Rd",t:"prop",c:100,col:"var(--sky)"},{n:"Pentonville",t:"prop",c:120,col:"var(--sky)"},
            {n:"Visiting",t:"visit",icon:"👀"},{n:"Pall Mall",t:"prop",c:140,col:"var(--pink)"},
            {n:"Electric Co",t:"prop",c:150,col:"var(--utility)"},{n:"Whitehall",t:"prop",c:140,col:"var(--pink)"},
            {n:"Northumber.",t:"prop",c:160,col:"var(--pink)"},{n:"Marylebone",t:"prop",c:200,col:"var(--station)"},
            {n:"Bow St",t:"prop",c:180,col:"var(--orange)"},{n:"Chest",t:"chest",icon:"📦"},
            {n:"Marlborough",t:"prop",c:180,col:"var(--orange)"},{n:"Vine St",t:"prop",c:200,col:"var(--orange)"},
            {n:"Free Parking",t:"free",icon:"🅿️"},{n:"Strand",t:"prop",c:220,col:"var(--red)"},
            {n:"Chance",t:"chance",icon:"❓"},{n:"Fleet St",t:"prop",c:220,col:"var(--red)"},
            {n:"Trafalgar Sq",t:"prop",c:240,col:"var(--red)"},{n:"Fenchurch",t:"prop",c:200,col:"var(--station)"},
            {n:"Leicester Sq",t:"prop",c:260,col:"var(--yellow)"},{n:"Coventry St",t:"prop",c:260,col:"var(--yellow)"},
            {n:"Water Works",t:"prop",c:150,col:"var(--utility)"},{n:"Piccadilly",t:"prop",c:280,col:"var(--yellow)"},
            {n:"GO TO JAIL",t:"jail",icon:"🚔"},{n:"Regent St",t:"prop",c:300,col:"var(--dgreen)"},
            {n:"Oxford St",t:"prop",c:300,col:"var(--dgreen)"},{n:"Chest",t:"chest",icon:"📦"},
            {n:"Bond St",t:"prop",c:320,col:"var(--dgreen)"},{n:"Liverpool St",t:"prop",c:200,col:"var(--station)"},
            {n:"Chance",t:"chance",icon:"❓"},{n:"Park Lane",t:"prop",c:350,col:"var(--navy)"},
            {n:"Super Tax",t:"tax",icon:"💸"},{n:"Mayfair",t:"prop",c:400,col:"var(--navy)"}
        ];

        // Player positions (simple state for this demo)
        let positions = [0, 0];

        function boardToGrid(i) {
            if (i<=10) return {row:10,col:10-i};
            if (i<=20) return {row:10-(i-10),col:0};
            if (i<=30) return {row:0,col:i-20};
            return {row:i-30,col:10};
        }
        function getSide(i) {
            if (i<=10) return "side-bottom";
            if (i<=20) return "side-left";
            if (i<=30) return "side-top";
            return "side-right";
        }

        // Build board
        const board = document.getElementById("board");
        for (let i=0;i<40;i++){
            const g=boardToGrid(i), sp=B[i], side=getSide(i);
            const isCorner=[0,10,20,30].includes(i);
            const div=document.createElement("div");
            div.className="cell "+side+(isCorner?" corner":"");
            div.style.gridRow=g.row+1;
            div.style.gridColumn=g.col+1;
            div.id="cell-"+i;
            let h="";
            if(sp.t==="prop"&&sp.col) h+='<div class="color-bar" style="background:'+sp.col+'"></div>';
            if(sp.icon) h+='<div class="icon">'+sp.icon+'</div>';
            h+='<div class="name">'+sp.n+'</div>';
            if(sp.c) h+='<div class="price">£'+sp.c+'</div>';
            h+='<div class="tokens" id="tokens-'+i+'"></div>';
            div.innerHTML=h;
            board.appendChild(div);
        }
        const center=document.createElement("div");
        center.className="board-center";
        center.innerHTML='<div class="title">MONOPOLY</div>';
        board.appendChild(center);

        // Update tokens on the board
        function updateTokens() {
            // Step 1: clear all token containers
            for (let i = 0; i < 40; i++) {
                const tc = document.getElementById("tokens-" + i);
                if (tc) tc.innerHTML = "";
            }

            // Step 2: place tokens in the correct cells
            for (let p = 0; p < 2; p++) {
                const tc = document.getElementById("tokens-" + positions[p]);
                if (tc) {
                    const tok = document.createElement("div");
                    tok.className = "token token-p" + (p + 1);
                    tok.textContent = p + 1;
                    tc.appendChild(tok);
                }
            }
        }

        // Move a player by a random dice roll
        function movePlayer(playerIndex) {
            const d1 = Math.floor(Math.random() * 6) + 1;
            const d2 = Math.floor(Math.random() * 6) + 1;
            const total = d1 + d2;
            positions[playerIndex] = (positions[playerIndex] + total) % 40;

            const name = "Player " + (playerIndex + 1);
            const space = B[positions[playerIndex]].n;
            document.getElementById("info").textContent =
                name + " rolled " + d1 + "+" + d2 + " = " + total + " → " + space;

            updateTokens();
        }

        // Initial render
        updateTokens();
    </script>
</body>
</html>
"""

filepath = os.path.join(project_folder, "p2_03_tokens.html")
with open(filepath, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Token demo saved to: {filepath}")
print("Click the buttons to move each player around the board!")

## Section 4 — How updateTokens Works

The function follows a simple two-step pattern:

**Step 1: Clear everything.** Loop through all 40 cells and empty their token containers. This is simpler and safer than trying to move individual tokens.

```javascript
for (let i = 0; i < 40; i++) {
    const tc = document.getElementById("tokens-" + i);
    if (tc) tc.innerHTML = "";
}
```

**Step 2: Place tokens.** Loop through the players, find the token container for their current position, and add a new token element.

```javascript
for (let p = 0; p < 2; p++) {
    const tc = document.getElementById("tokens-" + positions[p]);
    const tok = document.createElement("div");
    tok.className = "token token-p" + (p + 1);
    tok.textContent = p + 1;
    tc.appendChild(tok);
}
```

This "clear then rebuild" approach is used throughout web development. It avoids tricky bugs where tokens get left behind or duplicated.

## Section 5 — The CSS Transition

Notice this line in the token CSS:

```css
.token {
    transition: all 0.4s cubic-bezier(0.34, 1.56, 0.64, 1);
}
```

The `cubic-bezier(0.34, 1.56, 0.64, 1)` is a custom easing curve that creates a slight "bounce" effect. The `0.4s` is the duration. Although our current "clear and rebuild" approach does not trigger smooth movement between cells (the token jumps instantly), this transition will make the token's appearance feel alive when it pops into a cell.

## Section 6 — Challenge

1. Modify the demo so that both players start on different spaces (e.g., Player 1 on space 0, Player 2 on space 20). Check that tokens appear in the correct cells.
2. Add a third "Move Both" button that rolls dice for both players at once. Verify that when they land on the same space, both tokens appear side by side.
3. Add a counter below the info text that shows how many times each player has passed GO (hint: check if the new position is less than the old position).

In **Workshop 04** you will wire the full game logic from Phase 1 into this visual board. 🎲